In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import pickle

In [ ]:
## Load the dataset
data = pd.read_csv("./Churn_Modelling.csv")
print(data.shape)
print(data.head())
print(data.columns.tolist())

(10000, 14)
   RowNumber  CustomerId   Surname  ...  IsActiveMember EstimatedSalary Exited
0          1    15634602  Hargrave  ...               1       101348.88      1
1          2    15647311      Hill  ...               1       112542.58      0
2          3    15619304      Onio  ...               0       113931.57      1
3          4    15701354      Boni  ...               0        93826.63      0
4          5    15737888  Mitchell  ...               1        79084.10      0

[5 rows x 14 columns]
['RowNumber', 'CustomerId', 'Surname', 'CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited']


In [8]:
## preprocess the data
## drop the irrelevant variables
## RowNumber, CustomerId, Surname are irrelevant
data = data.drop(columns = ["RowNumber","CustomerId","Surname"], axis=1)
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [ ]:
## unique categorical values
print(data["Gender"].unique())
print(data["Geography"].unique())


['Female' 'Male']
['France' 'Spain' 'Germany']


In [13]:
## convert categorical values into numerical values
label_encoder_gender = LabelEncoder()
data["Gender"] = label_encoder_gender.fit_transform(data["Gender"])
print(data["Gender"].unique())

[0 1]


In [ ]:
## one hot encoding the geography column
## Frace, Spain, Germany
## any numerical value will weight one more than the others
## better to convert into OHE
from sklearn.preprocessing import OneHotEncoder
onehot_encoder_geo = OneHotEncoder()
data_geo = onehot_encoder_geo.fit_transform(data[["Geography"]])
data_geo ## we will get a sparse matrix with each row of size 3


<10000x3 sparse matrix of type '<class 'numpy.float64'>'
	with 10000 stored elements in Compressed Sparse Row format>

In [ ]:
## obtain the new column names
onehot_encoder_geo.get_feature_names_out(["Geography"])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [ ]:
data_geo.toarray()

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]])

In [ ]:
## drop the exisiting geography column
data = data.drop(columns=["Geography"],axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,0,42,2,0.00,1,1,1,101348.88,1
1,608,0,41,1,83807.86,1,0,1,112542.58,0
2,502,0,42,8,159660.80,3,1,0,113931.57,1
3,699,0,39,1,0.00,2,0,0,93826.63,0
4,850,0,43,2,125510.82,1,1,1,79084.10,0


In [39]:
## conver the data_geo array to pd dataframe before appending
## the column names are same as the array given by feature names
column_names = onehot_encoder_geo.get_feature_names_out(["Geography"])
data_geo_df = pd.DataFrame(data_geo.toarray(),columns=column_names)
data = pd.concat([data,data_geo_df],axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [41]:
## save the scalers and labeles
with open("label_encoder_gender.pkl","wb") as file:
    pickle.dump("label_encoder_gender",file)

with open("onehot_encoder_geo.pkl","wb") as file:
    pickle.dump("onehot_encoder_geo",file)

In [42]:
## divide the dataset into dependent and independent features
X=data.drop("Exited",axis=1) ## independent
y=data["Exited"] ## dependent

## train-test split (80-20)
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42,shuffle=True)


## scale the data using Scaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## never fit_transform() the test data
## otherwise it would be similar to leaking the answers


In [44]:
## save the scaled data
with open("scaler.pkl","wb") as file:
    pickle.dump("scaler",file)